# Reproduce Results
This notebook loads models from `models/`, runs predictions on the validation and test splits, and reports ROC AUC, precision and recall.
It must be fast to run and should not retrain models.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from utils import load_quora, split_quora, prepare_pair_texts, transform_tfidf, load_model
HOME = os.path.expanduser('~')
DATA_PATH = os.path.join(HOME, 'Datasets', 'QuoraQuestionPairs', 'quora_data.csv')

In [ ]:
df = load_quora(DATA_PATH)
train_df, val_df, test_df = split_quora(df, seed=123)
print('Shapes ->', train_df.shape, val_df.shape, test_df.shape)

In [ ]:
# Load vectorizer and model
VEC_PATH = os.path.join('models', 'tfidf_vectorizer.pkl')
MODEL_PATH = os.path.join('models', 'logreg_baseline.pkl')
vec = None
clf = None
if os.path.exists(VEC_PATH) and os.path.exists(MODEL_PATH):
    vec = load_model(VEC_PATH)
    clf = load_model(MODEL_PATH)
else:
    raise FileNotFoundError('Saved model or vectorizer not found in models/. Run train_models.ipynb first.')

In [ ]:
# Prepare texts and evaluate on validation and test sets
val_texts = prepare_pair_texts(val_df)
test_texts = prepare_pair_texts(test_df)
X_val = transform_tfidf(vec, val_texts)
X_test = transform_tfidf(vec, test_texts)
y_val = val_df['is_duplicate']
y_test = test_df['is_duplicate']
proba_val = clf.predict_proba(X_val)[:,1]
proba_test = clf.predict_proba(X_test)[:,1]
pred_val = (proba_val >= 0.5).astype(int)
pred_test = (proba_test >= 0.5).astype(int)
print('Val ROC AUC:', roc_auc_score(y_val, proba_val))
print('Val Precision:', precision_score(y_val, pred_val))
print('Val Recall:', recall_score(y_val, pred_val))
print('Test ROC AUC:', roc_auc_score(y_test, proba_test))
print('Test Precision:', precision_score(y_test, pred_test))
print('Test Recall:', recall_score(y_test, pred_test))